# 18. Semantic Segmentation 개념

이 노트북은 `17_탐지에서_분할로.ipynb` 다음 단계로, semantic segmentation의 기본 개념을 정리합니다.

Semantic segmentation은 이미지를 픽셀 단위로 분류하는 문제입니다. 이미지 전체에 클래스 하나를 붙이는 classification과 달리, semantic segmentation은 **모든 픽셀마다 class label을 예측**합니다.

이번 노트북의 목표는 다음과 같습니다.

- semantic segmentation을 픽셀 단위 classification으로 이해합니다.
- mask label, class map, color map의 차이를 구분합니다.
- semantic segmentation이 instance를 구분하지 않는다는 점을 이해합니다.
- segmentation 모델의 출력 텐서 형태를 직관적으로 파악합니다.


## 18-1. 준비

간단한 class map을 직접 만들고 색으로 시각화하면서 semantic segmentation label의 형태를 확인합니다.


In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False


## 18-2. 픽셀 단위 classification

Classification에서는 이미지 한 장에 대해 클래스 확률 벡터 하나를 출력합니다.

Semantic segmentation에서는 각 픽셀마다 클래스 확률 벡터를 출력합니다. 따라서 모델 출력은 보통 다음 형태를 가집니다.

```text
(batch, num_classes, height, width)
```

예를 들어 클래스가 `background`, `road`, `car`, `person` 네 개이고 출력 해상도가 `128 x 128`이라면, 한 이미지에 대한 출력은 `(4, 128, 128)` 형태가 됩니다.


In [ ]:
batch_size = 2
num_classes = 4
height, width = 128, 128

logits_shape = (batch_size, num_classes, height, width)
target_shape = (batch_size, height, width)

print('모델 출력 logits shape:', logits_shape)
print('정답 class map shape:', target_shape)
print('\n각 픽셀 위치마다 num_classes개의 점수가 있고, 정답은 픽셀마다 클래스 id 하나입니다.')


## 18-3. Class map 만들기

Semantic segmentation의 정답 mask는 보통 이미지와 같은 높이와 너비를 가진 2차원 배열입니다. 배열의 각 값은 해당 픽셀의 class id를 의미합니다.

아래 예시는 네 클래스를 사용합니다.

- `0`: background
- `1`: road
- `2`: car
- `3`: person


In [ ]:
class_names = {
    0: 'background',
    1: 'road',
    2: 'car',
    3: 'person',
}

class_map = np.zeros((12, 16), dtype=np.int64)
class_map[7:, :] = 1          # road
class_map[5:8, 3:8] = 2       # car 1
class_map[6:9, 10:14] = 2     # car 2
class_map[3:7, 8:10] = 3      # person

print(class_map)


숫자 배열만 보면 의미를 빠르게 파악하기 어렵습니다. 그래서 segmentation 결과를 볼 때는 class id를 색으로 바꾼 color map을 자주 사용합니다.


In [ ]:
colors = [
    '#d9d9d9',  # background
    '#4c78a8',  # road
    '#f58518',  # car
    '#54a24b',  # person
]
cmap = ListedColormap(colors)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(class_map, cmap=cmap, vmin=0, vmax=len(colors) - 1)
ax.set_title('Semantic segmentation class map')
ax.set_xticks(range(class_map.shape[1]))
ax.set_yticks(range(class_map.shape[0]))
ax.grid(color='white', linewidth=0.5)

handles = [plt.Line2D([0], [0], marker='s', color='w', markerfacecolor=colors[i], markersize=10, label=name)
           for i, name in class_names.items()]
ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.28, 1.0))
plt.show()


## 18-4. Mask label과 color map은 다르다

학습에 사용하는 정답은 보통 색 이미지가 아니라 class id가 들어 있는 배열입니다. 색은 사람이 보기 위한 시각화 방식입니다.

- mask label 또는 class map: 모델 학습과 평가에 사용하는 정답 배열
- color map: class id를 사람이 보기 쉬운 색으로 바꾼 이미지

따라서 segmentation 데이터셋을 다룰 때는 `색으로 보이는 값`과 `실제 class id`를 구분해야 합니다.


In [ ]:
unique_ids, counts = np.unique(class_map, return_counts=True)

for class_id, count in zip(unique_ids, counts):
    print(f'class {class_id} ({class_names[class_id]}): {count} pixels')


## 18-5. Semantic은 개별 객체를 구분하지 않는다

Semantic segmentation은 같은 클래스에 속한 픽셀을 같은 label로 표시합니다. 위 예시에서 자동차가 두 대 있어도 둘 다 class id `2`입니다.

즉, semantic segmentation의 관심은 다음 질문입니다.

```text
이 픽셀은 어떤 클래스인가?
```

반면 instance segmentation은 다음 질문까지 답해야 합니다.

```text
이 픽셀은 어떤 클래스이며, 몇 번째 객체에 속하는가?
```

따라서 같은 이미지에서 자동차가 두 대 있을 때 semantic segmentation은 `car 영역`만 표시하고, instance segmentation은 `car #1`, `car #2`를 따로 구분합니다.


## 18-6. 모델 출력에서 class map으로

Segmentation 모델은 각 픽셀마다 클래스별 점수(logit)를 냅니다. 최종 class map은 각 픽셀에서 가장 점수가 높은 클래스를 선택해서 만들 수 있습니다.


In [ ]:
rng = np.random.default_rng(0)
logits = rng.normal(size=(num_classes, 4, 5))
predicted_class_map = logits.argmax(axis=0)

print('logits shape:', logits.shape)
print('predicted class map shape:', predicted_class_map.shape)
print(predicted_class_map)


실제 학습에서는 이 logits와 정답 class map을 비교해 픽셀 단위 cross entropy loss를 계산합니다. 이 관점에서 semantic segmentation은 이미지를 구성하는 모든 픽셀에 classification을 수행하는 문제라고 볼 수 있습니다.


## 18-7. 정리

- Semantic segmentation은 모든 픽셀을 클래스 중 하나로 분류하는 문제입니다.
- 정답은 보통 `(height, width)` 형태의 class map입니다.
- 모델 출력은 보통 `(num_classes, height, width)` 형태의 픽셀별 class logits입니다.
- Color map은 학습용 정답이 아니라 사람이 보기 위한 시각화입니다.
- Semantic segmentation은 같은 클래스의 여러 객체를 개별 instance로 구분하지 않습니다.
- 다음 노트북에서는 CNN 분류 모델을 픽셀 단위 예측 모델로 바꾼 FCN의 핵심 아이디어를 다룹니다.
